# HSI α-sweep with a Transformer backbone (SSFormer)

Reviewer #4 asked for validation with a modern (Transformer-based) HSI backbone on top of the 2018 3D-CNN we originally used. This notebook reruns the α-sweep experiment with a compact Spectral-Spatial Transformer (**SSFormer**) in the spirit of SpectralFormer~\citep{hong2022spectralformer} and SSTN~\citep{zhong2022sstn}:
- Tokenize each $5\times5$ spatial location as a spectral vector
- Linear embed → 64-dim, prepend CLS token
- 4-layer Transformer encoder, 4 heads, feed-forward dim 128
- CLS → classifier head

Same split / evaluation protocol as `hsi_alpha_sweep.ipynb` (250 train + 50/50 cal/test, 10 seeds × 5 datasets, α ∈ {0.05, 0.10, 0.15}, joint $(r, \text{bw})$ CV).

Output pkls go to `/content/drive/MyDrive/hsi_sstn_alpha_sweep/checkpoints/` so that they do not overlap with the 3D-CNN pkls. Method (E) can be rerun post-hoc via `method_E_colab.ipynb` pointing at this new dir.

**Runtime**: ~3–5 hours on a Colab T4 (SSFormer is ~2× slower than 3D-CNN per epoch; we reduce epochs to 60 to compensate). Resumable via per-seed pkl cache + Drive mirror.

## 1. Setup + GPU check

In [3]:
!pip install -q scikit-learn scipy matplotlib

import os, sys, time, gc, pickle, json, shutil
import numpy as np
import scipy.io as sio
import torch
print('python', sys.version.split()[0], '| torch', torch.__version__, '| cuda', torch.cuda.is_available())
if torch.cuda.is_available(): print('GPU:', torch.cuda.get_device_name(0))
!nvidia-smi 2>/dev/null | head -10

python 3.12.13 | torch 2.10.0+cu128 | cuda True
GPU: Tesla T4
Sun Apr 19 03:47:36 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   40C    P8             10W /   70W |       3MiB /  15360MiB |      0%      Default |


## 2. Mount Drive + paths (local primary + Drive mirror)

In [4]:
USE_DRIVE = True
if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)

SACP_GEOCP_DIR = '/content/drive/MyDrive/sacp_geocp'
OLD_DATA_DIR   = f'{SACP_GEOCP_DIR}/datasets'

LOCAL_DIR   = '/content/hsi_sstn_alpha_sweep'
CKPT_DIR    = f'{LOCAL_DIR}/checkpoints'
RESULTS_DIR = f'{LOCAL_DIR}/results'
FIG_DIR     = f'{LOCAL_DIR}/figures'
DATA_DIR    = OLD_DATA_DIR
for d in [LOCAL_DIR, CKPT_DIR, RESULTS_DIR, FIG_DIR]:
    os.makedirs(d, exist_ok=True)

DRIVE_MIRROR   = '/content/drive/MyDrive/hsi_sstn_alpha_sweep'
DRIVE_CKPT_DIR = f'{DRIVE_MIRROR}/checkpoints'
try: os.makedirs(DRIVE_CKPT_DIR, exist_ok=True)
except OSError: pass

# Restore cached pkls from Drive to local
def _restore():
    if not os.path.isdir(DRIVE_CKPT_DIR): return 0
    try: pkls = [f for f in os.listdir(DRIVE_CKPT_DIR) if f.endswith('.pkl')]
    except OSError: return 0
    n = 0
    for f in pkls:
        src, dst = f'{DRIVE_CKPT_DIR}/{f}', f'{CKPT_DIR}/{f}'
        if os.path.exists(dst): continue
        try: shutil.copy2(src, dst); n += 1
        except OSError as e: print(f'[restore] {f}: {e}')
    return n
print(f'[restore] copied {_restore()} cached pkls from Drive to local')

print(f'DATA_DIR : {DATA_DIR}')
print(f'CKPT_DIR : {CKPT_DIR}')

expected = [('ip', 'Indian_pines_corrected.mat'), ('pu', 'PaviaU.mat'),
            ('sa', 'Salinas_corrected.mat'), ('ksc', 'KSC.mat'),
            ('botswana', 'Botswana.mat')]
missing = [(d,f) for (d,f) in expected if not os.path.exists(f'{DATA_DIR}/{d}/{f}')]
print('Missing:', missing if missing else 'none')

Mounted at /content/drive
[restore] copied 13 cached pkls from Drive to local
DATA_DIR : /content/drive/MyDrive/sacp_geocp/datasets
CKPT_DIR : /content/hsi_sstn_alpha_sweep/checkpoints
Missing: none


## 3. Helpers — SSFormer backbone + APS / SACP / GeoCP (inherited from hsi_alpha_sweep)

In [5]:
import torch.nn as nn, torch.nn.functional as F, torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split, KFold
from scipy.spatial.distance import cdist
from scipy.signal import convolve2d
from scipy import stats as sstats

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device =', device)

# ------ SSFormer: compact Transformer HSI backbone ------
class SSFormer(nn.Module):
    """Spectral-Spatial Transformer for HSI pixel classification.
    Tokenizes a 5x5 spatial patch into 25 spectral-vector tokens, plus a CLS token.
    Inspired by SpectralFormer (Hong 2022) and SSTN (Zhong 2022)."""
    def __init__(self, n_bands, n_classes, patch_size=5, dim=64, depth=4, heads=4, ffn=128, dropout=0.1):
        super().__init__()
        self.patch_size = patch_size
        self.n_tokens = patch_size * patch_size
        self.spectral_embed = nn.Sequential(
            nn.Linear(n_bands, dim),
            nn.LayerNorm(dim),
        )
        self.cls_token = nn.Parameter(torch.randn(1, 1, dim) * 0.02)
        self.pos_embed = nn.Parameter(torch.randn(1, self.n_tokens + 1, dim) * 0.02)
        enc_layer = nn.TransformerEncoderLayer(d_model=dim, nhead=heads,
                                               dim_feedforward=ffn, dropout=dropout,
                                               batch_first=True, activation='gelu',
                                               norm_first=True)
        self.encoder = nn.TransformerEncoder(enc_layer, num_layers=depth)
        self.norm = nn.LayerNorm(dim)
        self.head = nn.Linear(dim, n_classes)

    def forward(self, x):
        # x: (B, 1, C_bands, P, P) to match existing data pipeline
        if x.dim() == 5: x = x.squeeze(1)
        B, C, P, _ = x.shape
        x = x.permute(0, 2, 3, 1).reshape(B, P*P, C)     # (B, P*P, C_bands)
        x = self.spectral_embed(x)                        # (B, P*P, dim)
        cls = self.cls_token.expand(B, -1, -1)
        x = torch.cat([cls, x], dim=1) + self.pos_embed   # (B, P*P+1, dim)
        x = self.encoder(x)
        x = self.norm(x[:, 0])
        return self.head(x)

# ------ CP helpers (identical to hsi_alpha_sweep) ------
def aps_scores(probs, labels=None, rng=None):
    rng = rng or np.random
    n, K = probs.shape
    si = np.argsort(-probs, axis=1)
    sp = np.take_along_axis(probs, si, axis=1)
    cs = np.cumsum(sp, axis=1)
    U = rng.uniform(0, 1, size=(n, K))
    ss = cs - sp * U
    scores = np.zeros_like(ss)
    for i in range(n):
        scores[i, si[i]] = ss[i]
    if labels is not None:
        return np.array([scores[i, int(labels[i])] for i in range(n)])
    return scores

def conformal_quantile(scores, alpha):
    n = len(scores)
    return float(np.quantile(scores, np.ceil((n+1)*(1-alpha))/n))

def coverage_and_size(psets, labels):
    cov = float(np.mean([int(labels[i]) in s for i, s in enumerate(psets)]))
    sz  = float(np.mean([len(s) for s in psets]))
    return cov, sz

def set_interval_score(psets, true_labels, alpha):
    total = sum(len(psets[i]) + (2.0/alpha) * (0 if int(true_labels[i]) in psets[i] else 1)
                for i in range(len(psets)))
    return float(total / len(psets))

def set_interval_score_vec(score_mat, q_vec, y_true, alpha):
    in_set = score_mat < q_vec[:, None]
    sizes  = in_set.sum(axis=1)
    covered = in_set[np.arange(len(y_true)), y_true.astype(int)]
    penalty = (2.0 / alpha) * (~covered).astype(np.float64)
    return float((sizes + penalty).mean())

def sscv(psets, y, alpha, buckets=((1,1),(2,2),(3,3),(4,4),(5,10**9))):
    sizes = np.array([len(s) for s in psets])
    covered = np.array([int(y[i]) in psets[i] for i in range(len(y))])
    worst = 0.0; rows = []
    for lo, hi in buckets:
        m = (sizes >= lo) & (sizes <= hi)
        if m.sum() == 0:
            rows.append((lo, hi, 0, None)); continue
        cov = float(covered[m].mean())
        rows.append((lo, hi, int(m.sum()), cov))
        worst = max(worst, abs(cov - (1 - alpha)))
    return float(worst), rows

def sacp_smooth_radius(score_map, H, W, valid_idx, radius, lmd=0.5, k_iter=1):
    N, K = score_map.shape
    if radius <= 0: return score_map.copy()
    mask = np.zeros(N, dtype=bool); mask[valid_idx] = True
    mask_2d = mask.reshape(H, W).astype(np.float64)
    score_2d = score_map.reshape(H, W, K).astype(np.float64)
    k_size = 2 * radius + 1
    kernel = np.ones((k_size, k_size), dtype=np.float64); kernel[radius, radius] = 0.0
    for _ in range(k_iter):
        den = convolve2d(mask_2d, kernel, mode='same', boundary='fill', fillvalue=0.0)
        smoothed = np.empty_like(score_2d)
        for cls in range(K):
            values = np.where(mask_2d > 0, score_2d[..., cls], 0.0)
            num = convolve2d(values, kernel, mode='same', boundary='fill', fillvalue=0.0)
            smoothed[..., cls] = np.where(den > 1e-10, num / np.maximum(den, 1e-10), 0.0)
        new_score = (1 - lmd) * score_2d + lmd * smoothed
        score_2d = np.where(mask_2d[..., None] > 0, new_score, score_2d)
    return score_2d.reshape(N, K)

def vectorised_weighted_quantile(sorted_scores, d_matrix, order, bw, alpha):
    log_w = -0.5 * (d_matrix / bw) ** 2
    log_w -= log_w.max(axis=1, keepdims=True)
    w = np.exp(log_w); w_sorted = w[:, order]
    ws = w_sorted / w_sorted.sum(axis=1, keepdims=True)
    cum = np.cumsum(ws, axis=1)
    k_star = np.argmax(cum >= (1 - alpha), axis=1)
    return sorted_scores[k_star]

def extract_patches(hsi_chw, indices, patch_size=2):
    d, h, w = hsi_chw.shape
    padded = np.pad(hsi_chw, ((0,0),(patch_size,patch_size),(patch_size,patch_size)), mode='reflect')
    patches = np.zeros((len(indices), 1, d, 2*patch_size+1, 2*patch_size+1), dtype=np.float32)
    for e, idx in enumerate(indices):
        r, c = idx // w, idx % w
        patches[e, 0] = padded[:, r:r+2*patch_size+1, c:c+2*patch_size+1]
    return patches

DATASET_CONFIG = {
    'ip':  ('ip/Indian_pines_corrected.mat', 'indian_pines_corrected',
            'ip/Indian_pines_gt.mat', 'indian_pines_gt', 16, 200, 'Indian Pines'),
    'pu':  ('pu/PaviaU.mat', 'paviaU', 'pu/PaviaU_gt.mat', 'paviaU_gt', 9, 103, 'Pavia University'),
    'sa':  ('sa/Salinas_corrected.mat', 'salinas_corrected',
            'sa/Salinas_gt.mat', 'salinas_gt', 16, 204, 'Salinas'),
    'ksc': ('ksc/KSC.mat', 'KSC', 'ksc/KSC_gt.mat', 'KSC_gt', 13, 176, 'KSC'),
    'botswana': ('botswana/Botswana.mat', 'Botswana',
                 'botswana/Botswana_gt.mat', 'Botswana_gt', 14, 145, 'Botswana'),
}
print('Helpers + SSFormer loaded.')

device = cuda
Helpers + SSFormer loaded.


## 4. α-sweep pipeline (Transformer variant)

In [6]:
ALPHA_GRID = (0.05, 0.10, 0.15)
ALPHA_CV   = 0.10
EPOCHS     = 60     # SSFormer converges faster than 3D-CNN on small-data HSI; also 2x slower per epoch
LR         = 5e-4

def run_alpha_sweep(data_name, seed, epochs=EPOCHS, lmd=0.5,
                     radius_grid=(1, 2, 3, 5, 10),
                     bw_grid=(3, 5, 7, 10, 15, 20, 30, 50, 100)):
    ckpt_path = f'{CKPT_DIR}/{data_name}_seed{seed}.pkl'
    if os.path.exists(ckpt_path):
        with open(ckpt_path, 'rb') as f: return pickle.load(f)

    torch.manual_seed(seed*100 + 42); np.random.seed(seed*100 + 42)
    rng = np.random.RandomState(seed*100 + 42)

    hf, hk, gf, gk, n_classes, bands, nice = DATASET_CONFIG[data_name]
    hsi = sio.loadmat(f'{DATA_DIR}/{hf}')[hk]
    gt  = sio.loadmat(f'{DATA_DIR}/{gf}')[gk]
    h, w, d = hsi.shape; N = h*w; K = n_classes
    hsi_norm = hsi.astype(np.float32)
    hsi_norm = (hsi_norm - hsi_norm.mean(axis=(0,1))) / (hsi_norm.max() + 1e-8)
    hsi_chw = hsi_norm.transpose(2, 0, 1)

    y_all = gt.reshape(N)
    labeled_idx = np.where(y_all > 0)[0]
    coords = np.column_stack([np.arange(N)//w, np.arange(N)%w]).astype(np.float32)

    rs = seed*100 + 42
    idx_tr, idx_tmp = train_test_split(range(len(labeled_idx)), train_size=250,
                                        stratify=y_all[labeled_idx]-1, random_state=rs)
    idx_ca, idx_te = train_test_split(idx_tmp, test_size=0.5,
                                       stratify=(y_all[labeled_idx]-1)[idx_tmp], random_state=rs)
    tr_gi, ca_gi, te_gi = labeled_idx[idx_tr], labeled_idx[idx_ca], labeled_idx[idx_te]
    y_tr = y_all[tr_gi]-1; y_ca = y_all[ca_gi]-1; y_te = y_all[te_gi]-1

    # ---- Train SSFormer ----
    patch_size = 2
    X_tr = extract_patches(hsi_chw, tr_gi, patch_size)
    X_ca = extract_patches(hsi_chw, ca_gi, patch_size)
    X_te = extract_patches(hsi_chw, te_gi, patch_size)

    model = SSFormer(bands, n_classes, patch_size=5).to(device)
    train_loader = DataLoader(TensorDataset(torch.FloatTensor(X_tr), torch.LongTensor(y_tr)),
                              batch_size=32, shuffle=True)
    opt = optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
    sched = optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)
    crit = nn.CrossEntropyLoss()
    for _ in range(epochs):
        model.train()
        for Xb, yb in train_loader:
            Xb, yb = Xb.to(device), yb.to(device)
            opt.zero_grad(); crit(model(Xb), yb).backward(); opt.step()
        sched.step()

    model.eval()
    def get_probs(X):
        loader = DataLoader(TensorDataset(torch.FloatTensor(X)), batch_size=256, shuffle=False)
        out = []
        with torch.no_grad():
            for (b,) in loader:
                out.append(torch.softmax(model(b.to(device)), dim=1).cpu().numpy())
        return np.concatenate(out)

    probs_ca = get_probs(X_ca); probs_te = get_probs(X_te)
    pred_te = np.argmax(probs_te, axis=1)
    acc = float(np.mean(pred_te == y_te))

    del X_tr, X_ca, X_te, model
    gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()

    # ---- APS + SACP-smoothed per r ----
    cal_all   = aps_scores(probs_ca, rng=rng)
    test_all  = aps_scores(probs_te, rng=rng)
    cal_true  = aps_scores(probs_ca, y_ca, rng=rng)
    coords_ca = coords[ca_gi]; coords_te = coords[te_gi]
    sm = np.zeros((N, K))
    for e, i in enumerate(ca_gi): sm[i] = cal_all[e]
    for e, i in enumerate(te_gi): sm[i] = test_all[e]
    valid_idx_all = np.concatenate([ca_gi, te_gi])

    fused_per_r = {r: sacp_smooth_radius(sm, h, w, valid_idx_all, radius=r, lmd=lmd, k_iter=1)
                   for r in radius_grid}
    fcu_per_r = {r: np.array([fused_per_r[r][ca_gi[e], int(y_ca[e])] for e in range(len(idx_ca))])
                 for r in radius_grid}
    ftu_per_r = {r: np.array([fused_per_r[r][te_gi[e]] for e in range(len(idx_te))])
                 for r in radius_grid}

    # ---- 2D joint CV at ALPHA_CV ----
    kf = KFold(n_splits=5, shuffle=True, random_state=42)
    cv_is_sacp  = {r: [] for r in radius_grid}
    cv_is_geocp = {(r, bw): [] for r in radius_grid for bw in bw_grid}
    BATCH_VAL = 500
    for f_tr_idx, f_val_idx in kf.split(np.arange(len(idx_ca))):
        y_cv_val = y_ca[f_val_idx]; n_val = len(f_val_idx)
        for r in radius_grid:
            fcu_tr = fcu_per_r[r][f_tr_idx]
            fcu_val_all = fused_per_r[r][ca_gi[f_val_idx]]
            q_fold = conformal_quantile(fcu_tr, ALPHA_CV)
            cv_is_sacp[r].append(
                set_interval_score_vec(fcu_val_all, np.full(n_val, q_fold), y_cv_val, ALPHA_CV))
            order_tr = np.argsort(fcu_tr); sorted_tr = fcu_tr[order_tr]
            q_bw = {bw: np.empty(n_val) for bw in bw_grid}
            for b0 in range(0, n_val, BATCH_VAL):
                b1 = min(b0 + BATCH_VAL, n_val)
                d_batch = cdist(coords_ca[f_val_idx[b0:b1]], coords_ca[f_tr_idx])
                for bw in bw_grid:
                    q_bw[bw][b0:b1] = vectorised_weighted_quantile(sorted_tr, d_batch, order_tr, bw, ALPHA_CV)
                del d_batch
            for bw in bw_grid:
                cv_is_geocp[(r, bw)].append(
                    set_interval_score_vec(fcu_val_all, q_bw[bw], y_cv_val, ALPHA_CV))

    cv_sacp_mean  = {r: float(np.mean(v))  for r, v in cv_is_sacp.items()}
    cv_geocp_mean = {k: float(np.mean(v)) for k, v in cv_is_geocp.items()}
    best_r_sacp = int(min(cv_sacp_mean, key=cv_sacp_mean.get))
    best_rb     = min(cv_geocp_mean, key=cv_geocp_mean.get)
    best_r_geo, best_bw_geo = int(best_rb[0]), int(best_rb[1])

    # ---- Per-test GeoCP q at each α ----
    fcu_C = fcu_per_r[best_r_geo]; ftu_C = ftu_per_r[best_r_geo]
    order_C = np.argsort(fcu_C); sorted_C = fcu_C[order_C]
    BATCH_TEST = 1000; n_te = len(idx_te)
    q_C_per_alpha = {a: np.empty(n_te) for a in ALPHA_GRID}
    for b0 in range(0, n_te, BATCH_TEST):
        b1 = min(b0 + BATCH_TEST, n_te)
        dists = cdist(coords_te[b0:b1], coords_ca)
        for a in ALPHA_GRID:
            q_C_per_alpha[a][b0:b1] = vectorised_weighted_quantile(sorted_C, dists, order_C, best_bw_geo, a)
        del dists

    # ---- α loop: D / A / B / C ----
    per_alpha = {}
    for a in ALPHA_GRID:
        q_D = conformal_quantile(cal_true, a)
        ps_D = [np.where(test_all[i] < q_D)[0].tolist() for i in range(n_te)]
        q_A = conformal_quantile(fcu_per_r[1], a)
        ps_A = [np.where(ftu_per_r[1][i] < q_A)[0].tolist() for i in range(n_te)]
        q_B = conformal_quantile(fcu_per_r[best_r_sacp], a)
        ps_B = [np.where(ftu_per_r[best_r_sacp][i] < q_B)[0].tolist() for i in range(n_te)]
        q_C = q_C_per_alpha[a]
        ps_C = [np.where(ftu_C[i] < q_C[i])[0].tolist() for i in range(n_te)]
        row = {}
        for lab, ps in (('D', ps_D), ('A', ps_A), ('B', ps_B), ('C', ps_C)):
            cov, sz = coverage_and_size(ps, y_te)
            is_v    = set_interval_score(ps, y_te, a)
            sscv_v, sscv_rows = sscv(ps, y_te, a)
            row[lab] = {'cov': cov, 'size': sz, 'is': is_v, 'sscv_pct': sscv_v*100,
                        'sscv_per_bucket': sscv_rows, 'pred_sets': ps}
        row['q_C'] = q_C.tolist()
        per_alpha[f'{a:.2f}'] = row

    result = {
        'dataset': data_name, 'nice_name': nice, 'seed': int(seed),
        'backbone': 'SSFormer',
        'h': int(h), 'w': int(w), 'n_classes': int(n_classes),
        'bands': int(bands), 'lmd': float(lmd),
        'alpha_grid': list(ALPHA_GRID), 'alpha_cv': float(ALPHA_CV),
        'n_train': len(idx_tr), 'n_calib': len(idx_ca), 'n_test': len(idx_te),
        'accuracy': acc,
        'best_r_sacp': best_r_sacp,
        'best_r_geocp': best_r_geo, 'best_bw_geocp': best_bw_geo,
        'per_alpha': per_alpha,
        'cv_is_sacp': cv_sacp_mean,
        'cv_is_geocp': {f'{r}_{bw}': v for (r,bw), v in cv_geocp_mean.items()},
        'probs_ca': probs_ca.astype(np.float32), 'probs_te': probs_te.astype(np.float32),
        'cal_all_aps': cal_all.astype(np.float32), 'test_all_aps': test_all.astype(np.float32),
        'cal_true_aps': cal_true.astype(np.float32),
        'fcu_per_r':  {int(r): v.astype(np.float32) for r, v in fcu_per_r.items()},
        'ftu_per_r':  {int(r): v.astype(np.float32) for r, v in ftu_per_r.items()},
        'ca_gi': ca_gi.astype(np.int64), 'te_gi': te_gi.astype(np.int64),
        'y_ca': y_ca.astype(np.int64), 'y_te': y_te.astype(np.int64),
        'pred_te': pred_te.astype(np.int64),
    }
    tmp = ckpt_path + '.tmp'
    with open(tmp, 'wb') as f: pickle.dump(result, f, protocol=pickle.HIGHEST_PROTOCOL)
    os.replace(tmp, ckpt_path)
    return result

print('run_alpha_sweep (SSFormer) defined')

run_alpha_sweep (SSFormer) defined


## 5. Run 10 seeds × 5 datasets (resumable, Drive mirror)

In [7]:
# Defensive: ensure paths exist even if cell 4 was not re-run this session
LOCAL_DIR      = globals().get('LOCAL_DIR', '/content/hsi_sstn_alpha_sweep')
CKPT_DIR       = globals().get('CKPT_DIR', f'{LOCAL_DIR}/checkpoints')
RESULTS_DIR    = globals().get('RESULTS_DIR', f'{LOCAL_DIR}/results')
DRIVE_MIRROR   = globals().get('DRIVE_MIRROR', '/content/drive/MyDrive/hsi_sstn_alpha_sweep')
DRIVE_CKPT_DIR = globals().get('DRIVE_CKPT_DIR', f'{DRIVE_MIRROR}/checkpoints')
for _d in [LOCAL_DIR, CKPT_DIR, RESULTS_DIR]: os.makedirs(_d, exist_ok=True)
try: os.makedirs(DRIVE_CKPT_DIR, exist_ok=True)
except OSError: pass

DATASETS_TO_RUN = ['ip', 'pu', 'sa', 'ksc', 'botswana']
N_SEEDS = 10

log_file = f'{RESULTS_DIR}/run_log.txt'
total_runs = len(DATASETS_TO_RUN) * N_SEEDS
done = 0; t_start = time.time()

def _mirror_to_drive(ds, seed):
    src = f'{CKPT_DIR}/{ds}_seed{seed}.pkl'
    dst = f'{DRIVE_CKPT_DIR}/{ds}_seed{seed}.pkl'
    try:
        os.makedirs(DRIVE_CKPT_DIR, exist_ok=True); shutil.copy2(src, dst); return True
    except OSError: return False

for ds in DATASETS_TO_RUN:
    print(f'\n{"="*80}\n{DATASET_CONFIG[ds][6]}  ({ds})\n{"="*80}')
    for seed in range(N_SEEDS):
        ckpt = f'{CKPT_DIR}/{ds}_seed{seed}.pkl'
        if os.path.exists(ckpt):
            try:
                with open(ckpt, 'rb') as f: r = pickle.load(f)
            except OSError as e:
                print(f'  seed={seed} cached-read FAILED ({e}); will recompute.')
            else:
                done += 1
                pa = r['per_alpha']
                print(f'  seed={seed} [cached]  acc={r["accuracy"]:.3f}  '
                      f'α=0.10 IS: D={pa["0.10"]["D"]["is"]:.3f} A={pa["0.10"]["A"]["is"]:.3f} '
                      f'B={pa["0.10"]["B"]["is"]:.3f} C={pa["0.10"]["C"]["is"]:.3f}  '
                      f'(r*={r["best_r_geocp"]},bw*={r["best_bw_geocp"]})  [{done}/{total_runs}]')
                continue
        t0 = time.time()
        try:
            r = run_alpha_sweep(ds, seed)
            done += 1
            mirrored = _mirror_to_drive(ds, seed)
            pa = r['per_alpha']
            msg = (f'  seed={seed}           acc={r["accuracy"]:.3f}  '
                   f'α=0.10 IS: D={pa["0.10"]["D"]["is"]:.3f} A={pa["0.10"]["A"]["is"]:.3f} '
                   f'B={pa["0.10"]["B"]["is"]:.3f} C={pa["0.10"]["C"]["is"]:.3f}  '
                   f'(r*={r["best_r_geocp"]},bw*={r["best_bw_geocp"]})  '
                   f'[{time.time()-t0:.0f}s]  [{done}/{total_runs}]  '
                   f'{"mirrored" if mirrored else "(mirror skipped)"}')
            print(msg)
            try:
                with open(log_file, 'a') as f: f.write(f'{ds} {msg}\n')
            except OSError: pass
        except Exception as e:
            import traceback; print(f'  seed={seed} FAILED: {e}'); traceback.print_exc()

print(f'\n{"="*80}\nALL DONE: {done}/{total_runs} runs  ({(time.time()-t_start)/60:.1f} min)')


Indian Pines  (ip)
  seed=0 [cached]  acc=0.831  α=0.10 IS: D=3.654 A=3.364 B=3.327 C=3.169  (r*=10,bw*=10)  [1/50]
  seed=1 [cached]  acc=0.757  α=0.10 IS: D=3.860 A=3.528 B=3.352 C=3.044  (r*=5,bw*=7)  [2/50]
  seed=2 [cached]  acc=0.797  α=0.10 IS: D=3.974 A=3.562 B=3.340 C=3.176  (r*=5,bw*=5)  [3/50]
  seed=3 [cached]  acc=0.795  α=0.10 IS: D=3.794 A=3.546 B=3.382 C=3.291  (r*=10,bw*=7)  [4/50]
  seed=4 [cached]  acc=0.809  α=0.10 IS: D=3.732 A=3.293 B=3.178 C=3.074  (r*=5,bw*=20)  [5/50]
  seed=5 [cached]  acc=0.772  α=0.10 IS: D=3.833 A=3.246 B=3.127 C=3.009  (r*=10,bw*=10)  [6/50]
  seed=6 [cached]  acc=0.780  α=0.10 IS: D=3.907 A=3.403 B=3.318 C=3.150  (r*=5,bw*=5)  [7/50]
  seed=7 [cached]  acc=0.799  α=0.10 IS: D=3.926 A=3.488 B=3.321 C=3.150  (r*=10,bw*=7)  [8/50]
  seed=8 [cached]  acc=0.823  α=0.10 IS: D=3.721 A=3.363 B=3.288 C=3.140  (r*=10,bw*=7)  [9/50]
  seed=9 [cached]  acc=0.774  α=0.10 IS: D=3.989 A=3.427 B=3.272 C=3.021  (r*=5,bw*=5)  [10/50]

Pavia University  (p

/tmp/ipykernel_14620/1672960930.py:30: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.encoder = nn.TransformerEncoder(enc_layer, num_layers=depth)


  seed=3           acc=0.921  α=0.10 IS: D=3.179 A=3.007 B=2.985 C=2.883  (r*=5,bw*=15)  [760s]  [14/50]  mirrored


/tmp/ipykernel_14620/1672960930.py:30: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.encoder = nn.TransformerEncoder(enc_layer, num_layers=depth)


  seed=4           acc=0.933  α=0.10 IS: D=3.095 A=2.958 B=2.913 C=2.937  (r*=3,bw*=10)  [768s]  [15/50]  mirrored


...

  seed=7           acc=0.974  α=0.10 IS: D=3.019 A=2.919 B=2.898 C=3.031  (r*=10,bw*=50)  [17s]  [48/50]  mirrored


/tmp/ipykernel_14620/1672960930.py:30: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.encoder = nn.TransformerEncoder(enc_layer, num_layers=depth)


  seed=8           acc=0.955  α=0.10 IS: D=2.947 A=3.043 B=3.183 C=3.156  (r*=10,bw*=100)  [17s]  [49/50]  mirrored


/tmp/ipykernel_14620/1672960930.py:30: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.encoder = nn.TransformerEncoder(enc_layer, num_layers=depth)


  seed=9           acc=0.966  α=0.10 IS: D=3.093 A=3.105 B=2.885 C=3.155  (r*=1,bw*=100)  [17s]  [50/50]  mirrored

ALL DONE: 50/50 runs  (297.0 min)


## 6. Aggregate (same schema as 3D-CNN)

In [8]:
import pandas as pd
rows = []
for ds in DATASETS_TO_RUN:
    for seed in range(N_SEEDS):
        p = f'{CKPT_DIR}/{ds}_seed{seed}.pkl'
        if not os.path.exists(p): continue
        with open(p, 'rb') as f: r = pickle.load(f)
        for a_str, block in r['per_alpha'].items():
            a = float(a_str)
            for lab in ('D','A','B','C'):
                b = block[lab]
                rows.append({'backbone': 'SSFormer', 'dataset': ds, 'seed': seed, 'alpha': a, 'method': lab,
                             'cov': b['cov'], 'size': b['size'], 'is': b['is'], 'sscv_pct': b['sscv_pct'],
                             'accuracy': r['accuracy'],
                             'r_B': r['best_r_sacp'], 'r_C': r['best_r_geocp'], 'bw_C': r['best_bw_geocp']})
df = pd.DataFrame(rows)
df.to_csv(f'{RESULTS_DIR}/per_seed_alpha_sstn.csv', index=False)
print('Wrote', f'{RESULTS_DIR}/per_seed_alpha_sstn.csv  ({len(df)} rows)')

summary = (df.groupby(['alpha','method']).agg(
    cov_mean=('cov','mean'), size_mean=('size','mean'),
    is_mean=('is','mean'), sscv_mean=('sscv_pct','mean'),
    n=('seed','count')).reset_index())
print('\n=== Pooled SSFormer α-sweep ===')
print(summary.round(3).to_string(index=False))

from scipy import stats
print('\n=== C vs A / D paired tests per α (SSFormer) ===')
for a in sorted(df.alpha.unique()):
    for ref in ('A','D'):
        sub_r = df[(df.alpha==a)&(df.method==ref)].sort_values(['dataset','seed']).reset_index(drop=True)
        sub_c = df[(df.alpha==a)&(df.method=='C')].sort_values(['dataset','seed']).reset_index(drop=True)
        impr = (sub_r['is'].values - sub_c['is'].values) / sub_r['is'].values * 100
        t, p = stats.ttest_1samp(impr, 0.0)
        print(f'  α={a:.2f}  C vs {ref}: {impr.mean():+.2f}% ± {impr.std(ddof=1)/np.sqrt(len(impr)):.2f}  t={t:.2f}  p={p:.2e}  (n={len(impr)})')

Wrote /content/hsi_sstn_alpha_sweep/results/per_seed_alpha_sstn.csv  (600 rows)

=== Pooled SSFormer α-sweep ===
 alpha method  cov_mean  size_mean  is_mean  sscv_mean  n
  0.05      A     0.950      1.468    3.487      9.631 50
  0.05      B     0.950      1.322    3.319      8.627 50
  0.05      C     0.958      1.388    3.071      6.111 50
  0.05      D     0.950      1.842    3.843      7.159 50
  0.10      A     0.899      1.241    3.255     18.997 50
  0.10      B     0.899      1.172    3.201     14.427 50
  0.10      C     0.908      1.235    3.066     13.121 50
  0.10      D     0.899      1.457    3.474     11.588 50
  0.15      A     0.849      1.125    3.137     23.873 50
  0.15      B     0.848      1.082    3.109     17.709 50
  0.15      C     0.855      1.131    3.060     15.642 50
  0.15      D     0.849      1.268    3.276     17.277 50

=== C vs A / D paired tests per α (SSFormer) ===
  α=0.05  C vs A: +10.54% ± 1.48  t=7.13  p=4.19e-09  (n=50)
  α=0.05  C vs D: +18.

## 7. Package + push to Drive

In [9]:
import shutil
zip_local = '/content/hsi_sstn_alpha_sweep_results.zip'
shutil.make_archive('/content/hsi_sstn_alpha_sweep_results', 'zip', LOCAL_DIR)
try:
    os.makedirs(DRIVE_MIRROR, exist_ok=True)
    shutil.copy2(zip_local, f'{DRIVE_MIRROR}/hsi_sstn_alpha_sweep_results.zip')
    print('Copied zip to', f'{DRIVE_MIRROR}/hsi_sstn_alpha_sweep_results.zip')
except OSError as e:
    print(f'Drive copy failed ({e}); use files.download("{zip_local}")')

Copied zip to /content/drive/MyDrive/hsi_sstn_alpha_sweep/hsi_sstn_alpha_sweep_results.zip
